# SFT Step 02: Prepare v4 → Chat-Template JSONL

**Goal**: shape `datasets_v4/reasoning/cot_dataset.parquet` (3,926 rows = 1,963 fraud + 1,963 matched non-fraud, with `enhanced_prompt` + `cot_completion`) into chat-template JSONL the TRL `SFTTrainer` can consume in step 03.

**Inputs**
- `datasets_v4/huggingface/data/cot_reasoning/train.parquet` (or the equivalent `datasets_v4/reasoning/cot_dataset.parquet` — same 3,926 rows). Already balanced 50/50 fraud and matched on `(archetype, instrument, amount_band)`.

**Outputs** (written to Drive on Colab, or local `notebooks/sft_v3/data/` otherwise)
- `sft_train.jsonl`
- `sft_eval.jsonl`
- `sft_split_manifest.csv` — per-row record of which split each `data_uuid` landed in

**Format per line** (Gemma-compatible — only `user`/`assistant` roles; Gemma has no native `system` turn):
```json
{"messages": [{"role": "user", "content": "...enhanced_prompt..."}, {"role": "assistant", "content": "...cot_completion..."}], "meta": {"data_uuid": "...", "archetype": "...", "is_fraud": 0}}
```

Stratified 90/10 split on `(archetype × is_fraud)` so the eval set keeps the same per-archetype fraud balance as train.

In [ ]:
# --- 1. Mount Drive if running in Colab; otherwise stay local ---
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/drive/MyDrive/fraud-detection-framework')  # override if your Drive layout differs
    OUT_DIR   = Path('/content/drive/MyDrive/hf_cache/sft_v3')
else:
    REPO_ROOT = Path(r'C:\Nach\Fraud Detection Framework')
    OUT_DIR   = REPO_ROOT / 'notebooks' / 'sft_v3' / 'data'

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repo root: {REPO_ROOT}')
print(f'Output dir: {OUT_DIR}')

In [ ]:
# --- 2. Config ---
EVAL_FRAC = 0.10        # 90/10 stratified split
SEED      = 42

# Prefer the HF-export copy (the one we publish); fall back to the source under reasoning/
CANDIDATES = [
    REPO_ROOT / 'datasets_v4' / 'huggingface' / 'data' / 'cot_reasoning' / 'train.parquet',
    REPO_ROOT / 'datasets_v4' / 'reasoning' / 'cot_dataset.parquet',
]
SRC = next((p for p in CANDIDATES if p.exists()), None)
assert SRC is not None, f'No CoT parquet found at any of: {[str(p) for p in CANDIDATES]}'
print(f'Source: {SRC}')

In [ ]:
# --- 3. Load + sanity checks ---
import pandas as pd

df = pd.read_parquet(SRC)
print(f'Rows: {len(df)}')
print(f'Columns: {list(df.columns)}')

REQUIRED = ['data_uuid', 'archetype', 'is_fraud', 'enhanced_prompt', 'cot_completion', 'instrument', 'amount_band']
missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, f'Missing required columns: {missing}'

# No empty prompts/completions
for col in ['enhanced_prompt', 'cot_completion']:
    n_empty = df[col].isna().sum() + (df[col].astype(str).str.len() == 0).sum()
    assert n_empty == 0, f'{col} has {n_empty} empty rows'

print('\nis_fraud distribution:')
print(df['is_fraud'].value_counts())
print('\nArchetype × is_fraud:')
print(df.groupby(['archetype', 'is_fraud']).size().unstack(fill_value=0))

In [ ]:
# --- 4. Verify the matched-negatives invariant (archetype, instrument, amount_band) ---
# Each fraud row should have at least one non-fraud row sharing the same (archetype, instrument, amount_band).
key_cols = ['archetype', 'instrument', 'amount_band']
fraud_keys = set(map(tuple, df.loc[df['is_fraud'] == 1, key_cols].itertuples(index=False, name=None)))
nonfraud_keys = set(map(tuple, df.loc[df['is_fraud'] == 0, key_cols].itertuples(index=False, name=None)))

unmatched_fraud = fraud_keys - nonfraud_keys
print(f'Distinct fraud keys: {len(fraud_keys)}')
print(f'Distinct non-fraud keys: {len(nonfraud_keys)}')
print(f'Fraud keys with no matched negative: {len(unmatched_fraud)}')
if unmatched_fraud:
    print('  (sample 5):', list(unmatched_fraud)[:5])

In [ ]:
# --- 5. Stratified 90/10 split on (archetype × is_fraud) ---
from sklearn.model_selection import train_test_split

df = df.reset_index(drop=True)
df['stratum'] = df['archetype'].astype(str) + '|' + df['is_fraud'].astype(str)

train_df, eval_df = train_test_split(
    df,
    test_size=EVAL_FRAC,
    stratify=df['stratum'],
    random_state=SEED,
)

print(f'Train: {len(train_df)}    Eval: {len(eval_df)}')
print('\nTrain archetype × is_fraud:')
print(train_df.groupby(['archetype', 'is_fraud']).size().unstack(fill_value=0))
print('\nEval archetype × is_fraud:')
print(eval_df.groupby(['archetype', 'is_fraud']).size().unstack(fill_value=0))

In [ ]:
# --- 6. Build chat-template messages and write JSONL ---
import json

def to_record(row):
    return {
        'messages': [
            {'role': 'user',      'content': row['enhanced_prompt']},
            {'role': 'assistant', 'content': row['cot_completion']},
        ],
        'meta': {
            'data_uuid':  row['data_uuid'],
            'archetype':  row['archetype'],
            'is_fraud':   int(row['is_fraud']),
            'instrument': row['instrument'],
            'amount_band': row['amount_band'],
        },
    }

def write_jsonl(frame, path):
    with open(path, 'w', encoding='utf-8') as f:
        for _, row in frame.iterrows():
            f.write(json.dumps(to_record(row), ensure_ascii=False) + '\n')
    return path

TRAIN_PATH = OUT_DIR / 'sft_train.jsonl'
EVAL_PATH  = OUT_DIR / 'sft_eval.jsonl'
write_jsonl(train_df, TRAIN_PATH)
write_jsonl(eval_df,  EVAL_PATH)
print(f'Wrote {TRAIN_PATH} ({TRAIN_PATH.stat().st_size/1e6:.2f} MB)')
print(f'Wrote {EVAL_PATH}  ({EVAL_PATH.stat().st_size/1e6:.2f} MB)')

# Manifest of which split each row went to (for reproducibility / leakage checks in step 04)
manifest = pd.concat([
    train_df[['data_uuid', 'archetype', 'is_fraud']].assign(split='train'),
    eval_df[['data_uuid', 'archetype', 'is_fraud']].assign(split='eval'),
])
MANIFEST_PATH = OUT_DIR / 'sft_split_manifest.csv'
manifest.to_csv(MANIFEST_PATH, index=False)
print(f'Wrote {MANIFEST_PATH}')

In [ ]:
# --- 7. Token-length stats with the Gemma tokenizer (informs max_seq_len in step 03) ---
# Loads the tokenizer cached by 01_download_gemma4.ipynb. Skips gracefully if not present.
TOKENIZER_DIR = None
if IN_COLAB:
    candidate = Path('/content/drive/MyDrive/hf_cache/models/google__gemma-4-e4b-it')
    if candidate.exists():
        TOKENIZER_DIR = candidate

if TOKENIZER_DIR is None:
    print('Gemma tokenizer not found locally — skipping token-length stats.')
else:
    from transformers import AutoTokenizer
    import numpy as np

    tok = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR))

    def msg_tokens(row):
        msgs = [
            {'role': 'user',      'content': row['enhanced_prompt']},
            {'role': 'assistant', 'content': row['cot_completion']},
        ]
        # apply_chat_template returns the full templated string; count its tokens
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        return len(tok(text, add_special_tokens=False)['input_ids'])

    sample = df.sample(min(500, len(df)), random_state=SEED)
    lens = sample.apply(msg_tokens, axis=1).to_numpy()
    for q in [50, 75, 90, 95, 99, 100]:
        print(f'p{q:>3} tokens: {int(np.percentile(lens, q))}')
    print(f'mean: {lens.mean():.0f}   max: {lens.max()}')
    print('\nUse the p99 or max as the floor when picking max_seq_len in 03_train_lora.')

In [ ]:
# --- 8. Verify outputs: line counts + first record ---
import json

def head(path, n=1):
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= n: break
            yield json.loads(line)

for path in [TRAIN_PATH, EVAL_PATH]:
    with open(path, 'r', encoding='utf-8') as f:
        n = sum(1 for _ in f)
    print(f'{path.name}: {n} lines')

first = next(head(TRAIN_PATH, 1))
print('\nFirst train record:')
print('  meta:', first['meta'])
print('  user[0:200]:', first['messages'][0]['content'][:200])
print('  assistant[0:200]:', first['messages'][1]['content'][:200])

## Next steps

- `03_train_lora.ipynb` — QLoRA fine-tune Gemma 4 E4B on `sft_train.jsonl`, evaluate loss on `sft_eval.jsonl`. Set `max_seq_len` from the p99 token length printed above.
- `04_eval_judge.ipynb` — generate verdicts on `sft_eval.jsonl`, score F1 against `meta.is_fraud`, judge CoT coherence per archetype.